In [1]:
import pandas as pd
import numpy as np
import re
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 
import string
import torch
import re
import random, os
import emoji

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, BertModel, BertTokenizer, AutoModelForSeq2SeqLM
from torch.utils.data import Dataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers import AutoModelForSequenceClassification, AutoConfig, BertForSequenceClassification


import torch.nn as nn
from transformers import EarlyStoppingCallback
import transformers
from transformers import Trainer, TrainingArguments
from datasets import Dataset as HFDataset
from sklearn.metrics import classification_report

import nltk
import nlpaug.augmenter.sentence as nas
import nlpaug.augmenter.word as naw
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from nlpaug.augmenter.word import BackTranslationAug

D:\Terminal\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\Terminal\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\Terminal\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The 8-bit optimizer is not available on your device, only available on CUDA for now.


In [2]:
df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Magang dan Lisensi\BoothCamp\Proyek NLP\scrapped_googleplay_PGNMobile.csv")
df.head()

,userName,score,at,content
0,Dona Agung,5,2026-05-06 13:03:11,Aman dan Nyaman
1,Griselda Hanania,5,2026-05-06 13:02:57,bagus sih
2,anas 17,5,2026-05-06 13:02:35,"sudah cukup baik, lumayan sebagai alternatif k..."
3,Ajeng Giovani,5,2026-05-06 12:56:32,"pelayanan oke, tingkatkan"
4,Satria Agung,5,2026-05-06 12:54:37,rekomendasi


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    s = text

    # 1. Lowercase
    s = s.lower()
    # 2. Hapus URL, email, mention, hashtag (atau kamu bisa keep hashtag jika penting)
    s = re.sub(r'http\S+|www\.\S+', ' ', s)
    s = re.sub(r'\S+@\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'#\w+', ' ', s)
    # 3. Hapus HTML tags
    s = re.sub(r'<.*?>', ' ', s)
    # 4. Ubah repeated punctuation/char (mis. "sooooo" -> "soo" atau biarkan satu)
    s = re.sub(r'(.)\1{2,}', r'\1\1', s)
    # 5. Hapus extra whitespace & strip
    s = re.sub(r'\s+', ' ', s).strip()
    # 6. convert emoji → teks
    s = emoji.demojize(s, delimiters=(" ", " "))  

    return s

In [5]:
df['content'] = df['content'].apply(clean_text)
df.head()

,userName,score,at,content
0,Dona Agung,5,2026-05-06 13:03:11,aman dan nyaman
1,Griselda Hanania,5,2026-05-06 13:02:57,bagus sih
2,anas 17,5,2026-05-06 13:02:35,"sudah cukup baik, lumayan sebagai alternatif k..."
3,Ajeng Giovani,5,2026-05-06 12:56:32,"pelayanan oke, tingkatkan"
4,Satria Agung,5,2026-05-06 12:54:37,rekomendasi


In [6]:
def load_kamus(file_path):
    kamus={}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                slang = parts[0]
                normal = " ".join(parts[1:])
                kamus[slang] = normal
    return kamus

In [7]:
def normalisasi_teks(teks, kamus):
    words = teks.split()
    return " ".join([kamus.get(w, w) for w in words])

In [8]:
kamus = load_kamus(r"C:\Users\Aditya P J\Documents\Kuliah\Skripsi\Data\kbba.txt")
df['content'] = df['content'].apply(lambda x: normalisasi_teks(x, kamus))
df.head(20)

,userName,score,at,content
0,Dona Agung,5,2026-05-06 13:03:11,aman dan nyaman
1,Griselda Hanania,5,2026-05-06 13:02:57,bagus sih
2,anas 17,5,2026-05-06 13:02:35,"sudah cukup baik, lumayan sebagai alternatif k..."
3,Ajeng Giovani,5,2026-05-06 12:56:32,"pelayanan oke, tingkatkan"
4,Satria Agung,5,2026-05-06 12:54:37,rekomendasi
5,Wayan Abdullah,5,2026-05-06 12:52:59,mempermudah transportasi
6,Takwa ansar Makatita,5,2026-05-06 12:50:04,mantapp untuk perjalannyaa
7,Ling Ny,1,2026-05-06 12:01:41,aplikasi sampah
8,Adi Purnomo,5,2026-05-06 11:58:31,bagus
9,ㄚㄖᗪ丨 丿尺7,1,2026-05-06 11:54:58,"tolong ditambah event yang bisa dapat coin, da..."


## Labeling

In [10]:

# Model RoBERTa Sentiment (contoh pakai indobert-base-p1 untuk sentiment)
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"   # bisa diganti dengan model sentimen Indonesia lain
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

D:\Terminal\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [11]:
#Fungsi Prediksi Sentimen

from tqdm import tqdm
tqdm.pandas()  # aktifkan tqdm untuk pandas

# Fungsi Prediksi Sentimen
def get_sentiment(text):

    if not isinstance(text, str) or text.strip() == "":
        return "EMPTY"

    result = sentiment_pipeline(
        text,
        truncation=True,
        max_length=512
    )[0]

    return result['label']
# apply dengan progress bar
df["sentiment"] = df["content"].progress_apply(get_sentiment)

df.head(30)


100%|████████████████████████████████████████████████████████████████████████████| 10000/10000 [11:50<00:00, 14.07it/s]


,userName,score,at,content,sentiment
0,Dona Agung,5,2026-05-06 13:03:11,aman dan nyaman,positive
1,Griselda Hanania,5,2026-05-06 13:02:57,bagus sih,positive
2,anas 17,5,2026-05-06 13:02:35,"sudah cukup baik, lumayan sebagai alternatif k...",positive
3,Ajeng Giovani,5,2026-05-06 12:56:32,"pelayanan oke, tingkatkan",positive
4,Satria Agung,5,2026-05-06 12:54:37,rekomendasi,positive
5,Wayan Abdullah,5,2026-05-06 12:52:59,mempermudah transportasi,positive
6,Takwa ansar Makatita,5,2026-05-06 12:50:04,mantapp untuk perjalannyaa,neutral
7,Ling Ny,1,2026-05-06 12:01:41,aplikasi sampah,negative
8,Adi Purnomo,5,2026-05-06 11:58:31,bagus,positive
9,ㄚㄖᗪ丨 丿尺7,1,2026-05-06 11:54:58,"tolong ditambah event yang bisa dapat coin, da...",neutral


In [55]:
#Label Encoding
label_map = {"positive": 2, "negative": 0, "neutral": 1}
df["sentiment_num"] = df["sentiment"].map(label_map)
df_netral = df[df['sentiment_num'] == 2]
df_netral.head(30)

,userName,score,at,content,sentiment,label_encoded,sentiment_num
9921,GUNTUR PUTRAWAN,5,2026-03-09 02:42:07,wokee,positive,2,2
9922,Desi Nurtanti,5,2026-03-09 02:36:28,sangat berguna sekali,positive,2,2
9925,Rudi yanto,5,2026-03-09 01:04:24,"mudah penggunaannya dan driver selalu ramah,ny...",positive,2,2
9926,MR Hokky Indonesia Muh. Erf Dear,5,2026-03-09 00:44:37,di do'ain kedepan makin jaya tetep creatif. su...,positive,2,2
9927,Ari Daryanto46,4,2026-03-09 00:39:48,"saya harap kedepannya gojek makin baik lagi, d...",positive,2,2
9930,wismono Mono,4,2026-03-09 00:23:08,lumayan membantu kelancaran proses pengantaran...,positive,2,2
9934,Arif Widaya,5,2026-03-08 23:41:10,mantap bos,positive,2,2
9941,Fajar Syaifi,5,2026-03-08 23:10:54,perbanyak promo agar tidak kalah sama sebelah,positive,2,2
9943,Muhamad Dini,5,2026-03-08 23:00:39,sangat mempermudah perjalanan,positive,2,2
9947,Ardi Purnawirawan,5,2026-03-08 22:35:39,ok nyaman sat set,positive,2,2


## Modeling

In [23]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df['content'], 
    df['sentiment_num'], 
    test_size=0.2, 
    stratify=df['sentiment_num'], 
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5,
    stratify=y_temp, 
    random_state=42
)


In [25]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [27]:
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

## LSTM

In [30]:
model = Sequential([
    Embedding(max_words, 128, input_length=max_len),

    Bidirectional(LSTM(64)),

    Dropout(0.5),

    Dense(64, activation='relu'),

    Dense(3, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

D:\Terminal\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train_cat,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 17s 60ms/step - accuracy: 0.7275 - loss: 0.6495 - val_accuracy: 0.8831 - val_loss: 0.3368
Epoch 2/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - accuracy: 0.9113 - loss: 0.2553 - val_accuracy: 0.9038 - val_loss: 0.2905
Epoch 3/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - accuracy: 0.9500 - loss: 0.1616 - val_accuracy: 0.9006 - val_loss: 0.3078
Epoch 4/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - accuracy: 0.9675 - loss: 0.1101 - val_accuracy: 0.8900 - val_loss: 0.3502
Epoch 5/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - accuracy: 0.9739 - loss: 0.0833 - val_accuracy: 0.8950 - val_loss: 0.4233


In [33]:
loss, acc = model.evaluate(X_test_pad, y_test_cat)

print("Accuracy:", acc)

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8941 - loss: 0.2911
Accuracy: 0.8980000019073486


In [34]:
def predict_sentiment(text):

    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(padded)

    label = np.argmax(pred)

    sentiment = le.inverse_transform([label])[0]

    return sentiment

contoh = "Aplikasinya sangat membantu dan cepat"

hasil = predict_sentiment(contoh)

print("Hasil Sentimen:", hasil)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 394ms/step
Hasil Sentimen: positive


## RNN

In [47]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    Dense,
    Dropout
)

from gensim.models import Word2Vec

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [67]:
num_classes = len(np.unique(y_train))

# =========================================================
# WORD2VEC
# =========================================================
sentences = [text.split() for text in X_train]

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

# =========================================================
# TOKENIZER
# =========================================================
max_words = 10000
max_len = 64

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding='post'
)

# =========================================================
# EMBEDDING MATRIX
# =========================================================
embedding_dim = 100

embedding_matrix = np.zeros(
    (max_words, embedding_dim)
)

for word, i in tokenizer.word_index.items():

    if i >= max_words:
        continue

    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

# =========================================================
# MODEL SIMPLE RNN
# =========================================================
model = Sequential([

    Embedding(
        input_dim=max_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=False
    ),

    SimpleRNN(64),

    Dropout(0.5),

    Dense(32, activation='relu')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# EARLY STOPPING
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# TRAINING
history = model.fit(

    X_train_pad,
    y_train,

    validation_split=0.2,

    epochs=10,

    batch_size=32,

    callbacks=[early_stop]
)

# =========================================================
# EVALUASI
# =========================================================
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("\nAccuracy:", accuracy)

# =========================================================
# PREDIKSI
# =========================================================
y_pred = model.predict(X_test_pad)

y_pred_class = np.argmax(y_pred, axis=1)

y_true = y_test

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred_class,
        target_names=le.classes_
    )
)

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)              │ ?                           │       1,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_8 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_9 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.3986 - loss: 3.1698 - val_accuracy: 0.4625 - val_loss: 1.3475
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4804 - loss: 1.2836 - val_accuracy: 0.4625 - val_loss: 1.3475
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4741 - loss: 1.3036 - val_accuracy: 0.4625 - val_loss: 1.3460
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4760 - loss: 1.3566 - val_accuracy: 0.4625 - val_loss: 1.3451
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4889 - loss: 1.3362 - val_accuracy: 0.5031 - val_loss: 1.0810
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4867 - loss: 1.2281 - val_accuracy: 0.4619 - val_loss: 1.3448
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4734 - loss: 1.1585 - val_accuracy: 0.4575 - val_loss: 0.8486
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5006 - loss: 1.1390 - val_accuracy:

D:\Terminal\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Terminal\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Terminal\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## TF IDF SVM

In [73]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import LabelEncoder

from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix)

In [75]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)


In [81]:

# MODEL SVM
model = LinearSVC()

# TRAINING
model.fit(
    X_train_tfidf,
    y_train
)

y_pred = model.predict(X_test_tfidf)


# EVALUASI
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:", accuracy)

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=le.classes_
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


Accuracy: 0.902

Classification Report:

              precision    recall  f1-score   support

    negative       0.91      0.93      0.92       480
     neutral       0.43      0.15      0.22        41
    positive       0.91      0.94      0.92       479

    accuracy                           0.90      1000
   macro avg       0.75      0.67      0.69      1000
weighted avg       0.89      0.90      0.89      1000


Confusion Matrix:

[[448   4  28]
 [ 20   6  15]
 [ 27   4 448]]


In [85]:
!pip freeze > requirements.txt